In [2]:
from embedder import Embedder

embed = Embedder()

# q1 = "Can I still join the course after the start date?"
# q2 = "How to install Docker on Windows?"
# d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

# v1 = embed.encode(q1)
# v2 = embed.encode(q2)
# dv = embed.encode(d)

## Q1. Embedding a Query

In [3]:
query = "How does approximate nearest neighbor search work?"

v1 = embed.encode(query)

In [5]:
v1[0]

np.float64(-0.020582036807885073)

## Q2. Cosine Similarity

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [9]:
# Take Documents '02-vector-search/lessons/07-sqlitesearch-vector.md' and check the cosine similarity with Q1
for file in documents:
    if file["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        content = file["content"]
        break

In [10]:
content_vector = embed.encode(content)

In [ ]:
cosine_similarity = v1.dot(content_vector)

In [14]:
cosine_similarity

np.float64(0.361070280302606)

## Q3. Chunking and search by hand

In [17]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [24]:
chunks_content = [chunk["content"] for chunk in chunks]
X = embed.encode_batch(chunks_content)

In [25]:
# Get scores
scores = X.dot(v1)

In [27]:
import numpy as np

max_score = np.argmax(scores)
max_score

np.int64(94)

In [29]:
chunks[max_score]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

## Q4. Vector Search with minsearch